# Gaussian-splat decoder: same-target reconstruction report

This notebook separates the paper-faithful FuncBind baseline from Gaussian decoder replacement experiments on the exact same MCPP test molecule (`mcpp_dataset/1bm2/1bm2-CP.pdb`, target 0):

1. **Original pipeline:** checkpoint encoder + checkpoint INR decoder, with no fine-tuning.
2. **Maximum Gaussian overfit:** checkpoint encoder frozen and a fresh two-Gaussian-per-voxel decoder optimized with resampled occupied, tail, and background queries.
3. **Joint overfit ablation:** randomly initialized encoder and decoder optimized on one fixed sparse sample.
4. **Sparse frozen-encoder ablation:** checkpoint encoder frozen while a fresh decoder is optimized on the original fixed 4,096-point sample.

> Only the first condition is the original pretrained method. The maximum Gaussian run is a capacity test on one molecule, not held-out generalization.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    root for root in search_roots
    if (root / 'exps' / 'decoder_ablation').exists()
)
RESULT_PATH = (
    PROJECT_ROOT / 'exps' / 'decoder_ablation'
    / 'gpu2_fullgrid_validation_20260728' / 'comparison.json'
)
PRETRAINED_RESULT_PATH = (
    PROJECT_ROOT / 'exps' / 'decoder_ablation'
    / 'pretrained_frozen_mcpp_target0_20260728' / 'comparison.json'
)
ORIGINAL_RESULT_PATH = (
    PROJECT_ROOT / 'exps' / 'decoder_ablation'
    / 'original_pretrained_inr_mcpp_target0_20260728' / 'result.json'
)
MAX_GAUSSIAN_RESULT_PATH = (
    PROJECT_ROOT / 'exps' / 'decoder_ablation'
    / 'gaussian_max_overfit_mcpp_target0_20260728' / 'result.json'
)
with RESULT_PATH.open() as handle:
    report = json.load(handle)
with PRETRAINED_RESULT_PATH.open() as handle:
    pretrained_report = json.load(handle)
with ORIGINAL_RESULT_PATH.open() as handle:
    original_report = json.load(handle)
with MAX_GAUSSIAN_RESULT_PATH.open() as handle:
    max_gaussian_report = json.load(handle)
results = report['results']
pretrained_results = pretrained_report['results']
{
    'original checkpoint pipeline': ORIGINAL_RESULT_PATH,
    'maximum Gaussian overfit': MAX_GAUSSIAN_RESULT_PATH,
    'joint overfit ablation': RESULT_PATH,
    'frozen-encoder decoder adaptation': PRETRAINED_RESULT_PATH,
}

## Paper-faithful original FuncBind baseline

The checkpoint contains both `enc_state_dict` and `dec_state_dict`; these weights were trained together and must be loaded together. Reconstruction follows the original INR path with no weight optimization. The repository's original reconstruction defaults to a stochastic posterior sample; a deterministic posterior-mean render is included as a reproducibility control.

The comparison below now uses the maximum-overfit Gaussian result rather than the earlier sparse 7-atom result.

![Original pretrained pipeline versus maximum Gaussian overfit](figures/paper_faithful_original_pipeline_comparison.png)

In [ ]:
paper_rows = []
for mode in ('sample', 'mean'):
    result = original_report['results'][mode]
    reconstruction = result['reconstruction']
    atoms = result['atom_recovery']
    paper_rows.append({
        'method': f'Original pretrained INR · posterior {mode}',
        'full-grid MSE': reconstruction['density_mse'],
        'full-grid mIoU': reconstruction['miou'],
        'atom precision': atoms['atom_precision'],
        'atom recall': atoms['atom_recall'],
        'atom F1': atoms['atom_f1'],
        'coordinate RMSD (Å)': atoms['coordinate_rmsd'],
        'element accuracy': atoms['element_accuracy'],
        'predicted / target atoms': f"{atoms['n_predicted_atoms']} / {atoms['n_target_atoms']}",
    })
gaussian = max_gaussian_report['full_grid']
gaussian_atoms = gaussian['atom_recovery']
paper_rows.append({
    'method': 'Gaussian splat · maximum same-target overfit',
    'full-grid MSE': gaussian['reconstruction']['density_mse'],
    'full-grid mIoU': gaussian['reconstruction']['miou'],
    'atom precision': gaussian_atoms['atom_precision'],
    'atom recall': gaussian_atoms['atom_recall'],
    'atom F1': gaussian_atoms['atom_f1'],
    'coordinate RMSD (Å)': gaussian_atoms['coordinate_rmsd'],
    'element accuracy': gaussian_atoms['element_accuracy'],
    'predicted / target atoms': f"{gaussian_atoms['n_predicted_atoms']} / {gaussian_atoms['n_target_atoms']}",
})
pd.DataFrame(paper_rows).set_index('method').T

## Maximum Gaussian same-target overfit

The original fixed 4,096-point test batch contained only 20 points with occupancy ≥ 0.5. For this run, every step resampled occupied cores, Gaussian tails, and background from the complete $128^3$ target. Two Gaussians per latent voxel/channel were used because six carbon cell/channel bins contain two atoms.

Training ran for 5,000 steps and restored step 4,300, the best independent balanced-probe checkpoint. The resulting Gaussian decoder reaches full-grid MSE $5.75\times10^{-6}$, mIoU 0.844, and recovers all 57 atoms with 1.0 precision, recall, F1, and element accuracy. Coordinate RMSD is 0.262 Å.

![Maximum Gaussian overfit learning behavior](figures/gaussian_max_overfit_learning.png)

## Ablation A — jointly trained single-target overfit

![Sampled-point MSE and mIoU learning curves](figures/overfit_learning_curves.png)

This older diagnostic jointly optimized a randomly initialized encoder and decoder on the same molecule. It is useful for studying off-sample behavior, but its INR result is **not** the original FuncBind pipeline.

In [ ]:
colors = {'inr': '#64748b', 'gaussian_splat': '#e4572e'}
labels = {'inr': 'Fresh INR decoder', 'gaussian_splat': 'Fresh Gaussian decoder'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
for name in ('inr', 'gaussian_splat'):
    history = results[name]['history']
    steps = [point['step'] for point in history]
    axes[0].plot(steps, [point['density_mse'] for point in history],
                 marker='o', color=colors[name], label=labels[name])
    axes[1].plot(steps, [point['miou'] for point in history],
                 marker='o', color=colors[name], label=labels[name])
axes[0].set(yscale='log', xlabel='Optimization step', ylabel='Density MSE',
            title='Sampled-point reconstruction loss')
axes[1].set(xlabel='Optimization step', ylabel='mIoU',
            title='Sampled-point occupancy overlap', ylim=(-0.03, 1.05))
for axis in axes:
    axis.grid(alpha=0.22)
    axis.legend(frameon=False)
plt.show()

### Joint-overfit full-grid reconstruction

![Full-grid reconstruction, atom recovery, peak counts, and resource use](figures/overfit_fullgrid_summary.png)

The fresh Gaussian decoder generalized from sampled queries to the full grid better than the fresh INR decoder. This comparison describes the one-target ablation only; it must not be used as the paper baseline.

In [ ]:
rows = []
for name in ('inr', 'gaussian_splat'):
    result = results[name]
    full = result['full_grid']
    atoms = full['atom_recovery']
    rows.append({
        'decoder': labels[name],
        'sample MSE': result['final']['density_mse'],
        'sample mIoU': result['final']['miou'],
        'full-grid MSE': full['reconstruction']['density_mse'],
        'full-grid mIoU': full['reconstruction']['miou'],
        'atom precision': atoms['atom_precision'],
        'atom recall': atoms['atom_recall'],
        'atom F1': atoms['atom_f1'],
        'coordinate RMSD (Å)': atoms['coordinate_rmsd'],
        'element accuracy': atoms['element_accuracy'],
        'predicted / target atoms': f"{atoms['n_predicted_atoms']} / {atoms['n_target_atoms']}",
        'decoder parameters': result['decoder_parameters'],
        'mean step (s)': result['timing']['mean_step_seconds'],
        'peak training GPU (GiB)': result['timing']['peak_gpu_memory_gib'],
    })
pd.DataFrame(rows).set_index('decoder').T

## Maximum-overfit 3D occupancy result

This is the requested same-target occupancy view: reference, the complete original pretrained FuncBind INR pipeline, and the maximally overfit Gaussian decoder. At occupancy ≥ 0.1 the clouds contain 39,442 reference points, 39,325 original-INR points, and 39,981 Gaussian points.

![Reference, original pretrained INR, and maximum-overfit Gaussian occupancy](figures/overfit_occupancy_3d.png)

In [ ]:
ORIGINAL_OCCUPANCY_ROOT = ORIGINAL_RESULT_PATH.parent
occupancy_paths = {
    'Reference': ORIGINAL_OCCUPANCY_ROOT / 'reference_occupancy_points.npz',
    'Original pretrained INR': ORIGINAL_OCCUPANCY_ROOT / 'original_inr_sample_occupancy_points.npz',
    'Maximum Gaussian overfit': MAX_GAUSSIAN_RESULT_PATH.parent / 'gaussian_splat_occupancy_points.npz',
}

def load_occupancy_cloud(path):
    with np.load(path) as data:
        return {key: data[key] for key in data.files}

missing_occupancy_paths = {
    name: path for name, path in occupancy_paths.items() if not path.exists()
}
if missing_occupancy_paths:
    raise FileNotFoundError(missing_occupancy_paths)
occupancy_clouds = {
    name: load_occupancy_cloud(path) for name, path in occupancy_paths.items()
}
{name: cloud['coordinates'].shape[0] for name, cloud in occupancy_clouds.items()}

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

elements = ('C', 'O', 'N', 'S', 'F', 'Cl', 'P', 'Br')
element_colors = ('#4b5563', '#dc2626', '#2563eb', '#eab308', '#22c55e', '#16a34a', '#f97316', '#92400e')

def plot_occupancy_clouds(clouds, title):
    rng = np.random.default_rng(1234)
    figure = make_subplots(
        rows=1, cols=3, specs=[[{'type': 'scene'}] * 3],
        subplot_titles=list(clouds),
    )
    for column, (panel_title, cloud) in enumerate(clouds.items(), start=1):
        for channel, (element, color) in enumerate(zip(elements, element_colors)):
            indices = np.flatnonzero(cloud['channels'] == channel)
            if indices.size > 5000:
                indices = rng.choice(indices, 5000, replace=False)
            if not indices.size:
                continue
            xyz = cloud['coordinates'][indices]
            values = cloud['occupancies'][indices].astype(np.float32)
            figure.add_trace(
                go.Scatter3d(
                    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
                    mode='markers', name=element, legendgroup=element,
                    showlegend=(column == 1),
                    marker={'size': 1.5 + 3.5 * values, 'color': color, 'opacity': 0.36},
                    hovertemplate=(
                        f'{element}<br>x=%{{x:.2f}} Å<br>y=%{{y:.2f}} Å<br>'
                        'z=%{z:.2f} Å<extra></extra>'
                    ),
                ),
                row=1, col=column,
            )
    figure.update_scenes(
        aspectmode='cube',
        xaxis_title='x (Å)', yaxis_title='y (Å)', zaxis_title='z (Å)',
    )
    threshold = float(clouds['Reference']['threshold'])
    figure.update_layout(
        title=f'{title}: occupancy ≥ {threshold:.2f}',
        width=1500, height=620, margin={'l': 0, 'r': 0, 'b': 0, 't': 70},
    )
    return figure

figure_3d = plot_occupancy_clouds(occupancy_clouds, 'Original pretrained INR vs maximum Gaussian overfit')
figure_3d.show()

## Ablation B — sparse fresh decoders on the frozen pretrained encoder

This is the earlier fixed-query experiment, **not** the maximum overfit and not the original pipeline. The checkpoint encoder is frozen, but each decoder starts from random weights and sees the same fixed 4,096 uniform queries for 250 steps—only 20 of those queries have occupancy ≥ 0.5.

![Sparse fresh-decoder adaptation metrics](figures/overfit_pretrained_encoder_comparison.png)

Its 7-atom Gaussian result is retained only to demonstrate why balanced resampling was necessary.

In [ ]:
comparison_rows = []
for encoder_label, decoder_results in (
    ('Joint overfit: fresh encoder + decoder', results),
    ('Frozen pretrained encoder + fresh decoder', pretrained_results),
):
    for name in ('inr', 'gaussian_splat'):
        result = decoder_results[name]
        full = result['full_grid']
        atoms = full['atom_recovery']
        comparison_rows.append({
            'encoder': encoder_label,
            'decoder': labels[name],
            'sample MSE': result['final']['density_mse'],
            'sample mIoU': result['final']['miou'],
            'full-grid MSE': full['reconstruction']['density_mse'],
            'full-grid mIoU': full['reconstruction']['miou'],
            'atom precision': atoms['atom_precision'],
            'atom recall': atoms['atom_recall'],
            'atom F1': atoms['atom_f1'],
            'predicted atoms': atoms['n_predicted_atoms'],
            'matched atoms': atoms['matched_atoms'],
            'target atoms': atoms['n_target_atoms'],
        })
pd.DataFrame(comparison_rows).set_index(['encoder', 'decoder'])

### Frozen-encoder fresh-decoder occupancy ablation

These panels retain the earlier fresh-INR and fresh-Gaussian decoder results for diagnosis. They are separated from the corrected paper-faithful comparison above.

![Fresh decoders on frozen pretrained features](figures/overfit_pretrained_occupancy_3d.png)

In [ ]:
PRETRAINED_OCCUPANCY_ROOT = PRETRAINED_RESULT_PATH.parent
pretrained_occupancy_paths = {
    'Reference': PRETRAINED_OCCUPANCY_ROOT / 'reference_occupancy_points.npz',
    'Fresh INR decoder': PRETRAINED_OCCUPANCY_ROOT / 'inr_occupancy_points.npz',
    'Fresh Gaussian decoder': PRETRAINED_OCCUPANCY_ROOT / 'gaussian_splat_occupancy_points.npz',
}
missing_pretrained_paths = {
    name: path
    for name, path in pretrained_occupancy_paths.items()
    if not path.exists()
}
if missing_pretrained_paths:
    raise FileNotFoundError(missing_pretrained_paths)
pretrained_occupancy_clouds = {
    name: load_occupancy_cloud(path)
    for name, path in pretrained_occupancy_paths.items()
}
print({
    name: cloud['coordinates'].shape[0]
    for name, cloud in pretrained_occupancy_clouds.items()
})
plot_occupancy_clouds(
    pretrained_occupancy_clouds,
    'Fresh decoders on frozen pretrained encoder',
).show()

## Interpretation

- **Original method:** checkpoint encoder + checkpoint INR decoder yields full-grid mIoU 0.953 and 57/57 matched atoms at 0.258 Å RMSD.
- **Maximum Gaussian capacity:** the corrected Gaussian overfit yields mIoU 0.844 and also recovers 57/57 atoms, with 1.0 precision, recall, F1, and element accuracy at 0.262 Å RMSD.
- **Density gap:** Gaussian full-grid MSE is $5.75\times10^{-6}$ versus $5.16\times10^{-7}$ for original INR, so INR still reproduces the continuous density more accurately.
- **Cause of the earlier failure:** the fixed test sample had only 20 strongly occupied queries; that sparse run recovered only 7 atoms. Balanced resampling, two Gaussians per cell/channel, and 5,000 optimization steps remove the apparent capacity failure.
- **Scope:** this demonstrates that the Gaussian decoder can overfit the same frozen encoder latent for one molecule. It does not yet demonstrate dataset-level generalization.
- **Next fair experiment:** train the Gaussian decoder over the original neural-field training distribution with posterior sampling, augmentation, and a matched optimization budget, then evaluate the frozen Gaussian and INR checkpoints on held-out targets.